# Notebook 12: N-HiTS en PatchTST consumption huis
---
- Twee moderne deep-learningmodellen voor lange horizons, getraind op het verbruik (`consumption`) van het **huis**, 3 dagen (864 stappen van 5 minuten) vooruit.
- Zelfde opzet als GRU-consumption en notebook 11 (70/15/15 split, 7 dagen historie, testvensters elk uur), zodat de MAE direct vergelijkbaar is.
- Runtime (trainen en voorspellen) wordt per model bijgehouden met `time`.

### Literatuur: waarom deze modellen?
---
**Probleem met de eerdere modellen.** LSTM, GRU, SARIMAX en LightGBM bleven op ~900 W MAE steken, terwijl simpele persistence ("zelfde tijd gisteren") in notebook 06 op de laatste 20% van de reeks 363 W haalde. Dat wijst op een **distribution shift**: het niveau van het verbruik in 2026 wijkt af van de trainingsperiode. Een model dat op het gemiddelde en de standaarddeviatie van de trainingsset is genormaliseerd, zit dan structureel naast de werkelijkheid.

**Reversible Instance Normalization (RevIN)** — Kim et al. (2022) laten zien dat het normaliseren van *elk invoervenster afzonderlijk* (en de voorspelling daarna weer terugschalen) de fout bij distribution shift sterk verlaagt, voor verschillende modeltypes. Beide modellen hieronder gebruiken deze vorm van per-venster-normalisatie (PatchTST via `revin=True`, N-HiTS via `scaler_type="robust"`).

**N-HiTS** (Challu et al., 2023) is een opvolger van N-BEATS (Oreshkin et al., 2020): een stapel MLP-blokken die elk op een andere tijdschaal werken. Met *multi-rate pooling* kijkt het eerste blok naar grove patronen (bijv. per 6 uur), het laatste naar fijne details (5 minuten); de voorspellingen worden met *hierarchical interpolation* samengevoegd. De auteurs rapporteren op lange-horizonbenchmarks (o.a. de Electricity-dataset met huishoudelijk verbruik) gemiddeld bijna 20% nauwkeuriger resultaten dan de toen beste Transformers, bij een veelvoud kortere rekentijd. Dat past bij onze horizon van 864 stappen.

**PatchTST** (Nie et al., 2023) is een Transformer die de reeks opknipt in *patches* (stukjes van bijv. 4 uur) en die als "woorden" behandelt. Patching verkort de invoer voor de attention sterk en laat het model lokale vormen (een ochtend- of avondpiek) als geheel zien. Het kwam na Zeng et al. (2023), dat aantoonde dat eerdere Transformers op lange horizons vaak verloren van een simpel lineair model; PatchTST versloeg die lineaire modellen wel, ook op Electricity.

**Waarom allebei?** In de benchmarks is er geen vaste winnaar tussen de twee: het verschil hangt af van de dataset en de horizon. N-HiTS is sneller en eenvoudiger, PatchTST is krachtiger maar zwaarder. Daarom worden ze hier op exact dezelfde testset vergeleken.

**Verwachting.** Verbruik van een individueel huishouden is erg grillig en daardoor moeilijk te voorspellen (Kong et al., 2019); korte pieken (waterkoker, oven) zijn op 3 dagen vooruit niet voorspelbaar. Een realistische winst zit vooral in het goed treffen van niveau en dagpatroon.

**Bronnen**
- Challu, C., Olivares, K. G., Oreshkin, B. N., Garza, F., Mergenthaler-Canseco, M., & Dubrawski, A. (2023). N-HiTS: Neural Hierarchical Interpolation for Time Series Forecasting. *Proceedings of the AAAI Conference on Artificial Intelligence, 37*(6), 6989–6997.
- Kim, T., Kim, J., Tae, Y., Park, C., Choi, J.-H., & Choo, J. (2022). Reversible Instance Normalization for Accurate Time-Series Forecasting against Distribution Shift. *International Conference on Learning Representations (ICLR)*.
- Kong, W., Dong, Z. Y., Jia, Y., Hill, D. J., Xu, Y., & Zhang, Y. (2019). Short-Term Residential Load Forecasting Based on LSTM Recurrent Neural Network. *IEEE Transactions on Smart Grid, 10*(1), 841–851.
- Nie, Y., Nguyen, N. H., Sinthong, P., & Kalagnanam, J. (2023). A Time Series is Worth 64 Words: Long-term Forecasting with Transformers. *International Conference on Learning Representations (ICLR)*.
- Oreshkin, B. N., Carpov, D., Chapados, N., & Bengio, Y. (2020). N-BEATS: Neural basis expansion analysis for interpretable time series forecasting. *International Conference on Learning Representations (ICLR)*.
- Zeng, A., Chen, M., Zhang, L., & Xu, Q. (2023). Are Transformers Effective for Time Series Forecasting? *Proceedings of the AAAI Conference on Artificial Intelligence, 37*(9), 11121–11128.

### Inladen
---
1. Libraries en paths inladen
- Installatie (eenmalig): `pip install neuralforecast` (installeert ook PyTorch Lightning). Gebruikt automatisch de GPU als die er is.

In [ ]:
import time
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS, PatchTST
from neuralforecast.losses.pytorch import MAE

logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)
torch.set_float32_matmul_precision("high")

start_notebook = time.perf_counter()
runtimes = {}                                   # alle gemeten tijden (seconden)
print("GPU:", torch.cuda.is_available())

KANDIDATEN = [Path("../prepared_data/forecast_targets_with_weather.parquet"),
              Path("../../prepared_data/forecast_targets_with_weather.parquet")]
DATA_DIR = next(p for p in KANDIDATEN if p.exists())
RESULTS_DIR = DATA_DIR.parent.parent / "results"
RESULTS_DIR.mkdir(exist_ok=True)
print("Data:", DATA_DIR.resolve())

2. Voorspel instellingen
---

In [ ]:
stap_per_dag = 24 * 12                     # 288 stappen per dag (5 minuten)

aantal_dagen_input = 7                     # zelfde als GRU / LightGBM
input_len = aantal_dagen_input * stap_per_dag

aantal_dagen_forecast = 3
horizon = aantal_dagen_forecast * stap_per_dag   # 864 stappen

stride_test = 12                           # elk uur een nieuwe forecast (zelfde als GRU)

TRAIN_FRAC = 0.70
VAL_FRAC = 0.15

3. Data inladen en splitsen (zelfde filter en split als GRU-consumption)
---

In [ ]:
t0 = time.perf_counter()
df = pd.read_parquet(DATA_DIR)
huis_cons = df[(df["location"] == "huis") & (df["target_type"] == "consumption")].copy()
# zelfde filter als het GRU-consumption-notebook en notebook 11, zodat de testset identiek is
huis_cons = huis_cons.dropna(subset=["pow_imputed", "Windrichting", "WindSnelheid", "LuchtVochtigheid"])
huis_cons = huis_cons.sort_values("timestamp").reset_index(drop=True)
runtimes["data inladen"] = time.perf_counter() - t0

y_arr = huis_cons["pow_imputed"].to_numpy(np.float32)
n = len(y_arr)
i_train = int(n * TRAIN_FRAC)
i_val = int(n * (TRAIN_FRAC + VAL_FRAC))

# test: exact dezelfde forecastmomenten als GRU-consumption en notebook 11
test_origins = i_val + input_len + np.arange(0, (n - i_val) - input_len - horizon + 1, stride_test)

print(huis_cons.shape)
print(f"n = {n} | train t/m {i_train} | val t/m {i_val} | test t/m {n}")
print("Start test:", huis_cons["timestamp"].iloc[i_val])
print(f"Aantal test-forecasts: {len(test_origins)}")

4. Hulpfuncties voor evaluatie (baselines, scores, plots)
---

In [ ]:
def baselines(origins):
    """Persistence-baselines: zelfde tijdstip gisteren (meest recente bekende dag) en vorige week."""
    h = np.arange(horizon)
    t = origins[:, None] + h[None, :]
    k = h // stap_per_dag + 1
    dag = y_arr[t - stap_per_dag * k]
    week = y_arr[t - input_len]
    return dag, week


def echte_waarden(origins):
    return y_arr[origins[:, None] + np.arange(horizon)[None, :]]


def scores(pred, echt):
    fout = pred - echt
    res = {"MAE": np.abs(fout).mean(), "RMSE": np.sqrt((fout ** 2).mean())}
    for d in range(aantal_dagen_forecast):
        s = slice(d * stap_per_dag, (d + 1) * stap_per_dag)
        res[f"MAE dag {d+1}"] = np.abs(fout[:, s]).mean()
    # rolling validation: elke dag (stap 288) een nieuwe forecast, gemiddelde MAE/RMSE per venster
    dagelijks = slice(None, None, stap_per_dag // stride_test)
    res["rolling MAE"] = np.abs(fout[dagelijks]).mean(axis=1).mean()
    res["rolling RMSE"] = np.sqrt((fout[dagelijks] ** 2).mean(axis=1)).mean()
    return res


def plot_voorspelling(voorspellingen, idx=0, titel="Voorspelling vs. echt"):
    o = test_origins[idx]
    hist = y_arr[o - input_len:o]
    echt = y_arr[o:o + horizon]
    t_hist = np.arange(-input_len, 0) / stap_per_dag
    t_fc = np.arange(horizon) / stap_per_dag

    plt.figure(figsize=(12, 4))
    plt.plot(t_hist, hist, color="grey", label="Input (historie)")
    plt.plot(t_fc, echt, color="black", alpha=0.7, label="Echt")
    for naam, pred in voorspellingen.items():
        mae = np.abs(pred[idx] - echt).mean()
        plt.plot(t_fc, pred[idx], label=f"{naam} (MAE {mae:.0f} W)")
    plt.axvline(0, color="black", linestyle="--", alpha=0.5)
    plt.xlabel("Dagen t.o.v. start voorspelling")
    plt.ylabel("Verbruik (W)")
    plt.title(titel)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


def plot_mae_per_stap(voorspellingen, echt):
    plt.figure(figsize=(10, 3.5))
    for naam, pred in voorspellingen.items():
        plt.plot(np.arange(horizon) / stap_per_dag, np.abs(pred - echt).mean(axis=0), label=naam)
    plt.xlabel("Dagen vooruit")
    plt.ylabel("MAE (W)")
    plt.title("MAE per voorspelstap")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

5. Data in het formaat van `neuralforecast`
---
- `neuralforecast` verwacht een tabel met `unique_id`, `ds` (tijd) en `y`.
- Als tijd wordt de **positie** (0, 1, 2, ...) gebruikt met `freq=1`. Dat sluit aan bij de GRU (die ook op positie werkt) en voorkomt problemen met zomer-/wintertijd.
- Getraind wordt op train + val, waarbij `val_size` de validatieperiode apart houdt voor early stopping. De testset wordt niet gezien.

In [ ]:
train_val_df = pd.DataFrame({"unique_id": "huis", "ds": np.arange(i_val), "y": y_arr[:i_val]})
val_size = i_val - i_train
print(train_val_df.shape, "| val_size:", val_size)

### Modellen
---
**N-HiTS**
- `n_freq_downsample=[72, 12, 1]`: de drie blokken voorspellen op resp. 6-uurs, uurs- en 5-minutenresolutie.
- `n_pool_kernel_size=[16, 4, 1]`: hoeveel de invoer per blok wordt samengevat (grof → fijn).

**PatchTST**
- `patch_len=48` (4 uur) met `stride=24` (2 uur overlap): 7 dagen historie worden 83 patches.
- `revin=True`: per-venster-normalisatie (Kim et al., 2022).

Beide: MAE-loss (zelfde als de L1-loss van de GRU), early stopping op de validatieset.

In [ ]:
MAX_STEPS = 2000          # maximaal aantal trainingsstappen (early stopping stopt meestal eerder)
PATIENCE = 5              # aantal validatiechecks zonder verbetering voor early stopping
VAL_CHECK = 100           # elke 100 stappen valideren

nhits = NHITS(
    h=horizon,
    input_size=input_len,
    n_pool_kernel_size=[16, 4, 1],
    n_freq_downsample=[72, 12, 1],
    mlp_units=[[512, 512]] * 3,
    scaler_type="robust",               # per-venster-normalisatie (mediaan / IQR)
    loss=MAE(),
    learning_rate=1e-3,
    max_steps=MAX_STEPS,
    early_stop_patience_steps=PATIENCE,
    val_check_steps=VAL_CHECK,
    batch_size=32,
    windows_batch_size=256,
    random_seed=42,
    alias="N-HiTS",
)

patchtst = PatchTST(
    h=horizon,
    input_size=input_len,
    patch_len=48,
    stride=24,
    encoder_layers=3,
    n_heads=8,
    hidden_size=128,
    linear_hidden_size=256,
    dropout=0.2,
    revin=True,                         # RevIN (Kim et al., 2022)
    scaler_type="identity",
    loss=MAE(),
    learning_rate=1e-4,
    max_steps=MAX_STEPS,
    early_stop_patience_steps=PATIENCE,
    val_check_steps=VAL_CHECK,
    batch_size=32,
    windows_batch_size=128,             # lager zetten bij geheugenproblemen op de GPU
    random_seed=42,
    alias="PatchTST",
)

MODELLEN = {"N-HiTS": nhits, "PatchTST": patchtst}

### Trainen
---
- Elk model wordt apart getraind zodat de runtime per model gemeten kan worden.

In [ ]:
nf_modellen = {}
for naam, model in MODELLEN.items():
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start_totaal = time.perf_counter()                       # start tijd

    nf = NeuralForecast(models=[model], freq=1)
    nf.fit(df=train_val_df, val_size=val_size)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    runtimes[f"training {naam}"] = time.perf_counter() - start_totaal
    nf_modellen[naam] = nf
    print(f"{naam}: totale trainingstijd {runtimes[f'training {naam}']:.1f}s "
          f"({runtimes[f'training {naam}'] / 60:.1f} min)")

In [ ]:
# Trainings- en validatieloss (als neuralforecast die heeft bijgehouden)
fig, axes = plt.subplots(1, len(nf_modellen), figsize=(12, 3.5))
for ax, (naam, nf) in zip(np.atleast_1d(axes), nf_modellen.items()):
    m = nf.models[0]
    train_trajectory = getattr(m, "train_trajectories", [])
    valid_trajectory = getattr(m, "valid_trajectories", [])
    if train_trajectory:
        ax.plot(*zip(*train_trajectory), label="Train")
    if valid_trajectory:
        ax.plot(*zip(*valid_trajectory), label="Validatie")
    ax.set_title(naam)
    ax.set_xlabel("Stap")
    ax.set_ylabel("MAE (W)")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Voorspellen op de testset
---
- Elk testvenster (7 dagen historie) wordt als aparte reeks (`unique_id`) aangeboden; zo voorspelt het model alle 2600+ vensters in een paar batches.
- Het model gebruikt alleen de laatste `input_len` punten van elke reeks.

In [ ]:
def voorspel_nf(nf, naam, origins, blok=500):
    preds = []
    for i in range(0, len(origins), blok):
        deel = origins[i:i + blok]
        ctx = np.stack([y_arr[o - input_len:o] for o in deel])               # (blok, input_len)
        ctx_df = pd.DataFrame({
            "unique_id": np.repeat(np.arange(len(deel)), input_len),
            "ds": np.tile(np.arange(input_len), len(deel)),
            "y": ctx.ravel(),
        })
        out = nf.predict(df=ctx_df)
        if "unique_id" not in out.columns:
            out = out.reset_index()
        out = out.sort_values(["unique_id", "ds"])
        preds.append(out[naam].to_numpy().reshape(len(deel), horizon))
    return np.vstack(preds)


echt_test = echte_waarden(test_origins)
voorspellingen = {}
for naam, nf in nf_modellen.items():
    t0 = time.perf_counter()
    voorspellingen[naam] = voorspel_nf(nf, naam, test_origins)
    runtimes[f"voorspellen {naam}"] = time.perf_counter() - t0
    print(f"{naam}: {len(test_origins)} forecasts in {runtimes[f'voorspellen {naam}']:.1f}s "
          f"({runtimes[f'voorspellen {naam}'] / len(test_origins) * 1000:.1f} ms per forecast)")

base_dag, base_week = baselines(test_origins)

### Resultaten
---

In [ ]:
alle = {**voorspellingen, "Persistence dag": base_dag, "Persistence week": base_week}
resultaten = pd.DataFrame({naam: scores(pred, echt_test) for naam, pred in alle.items()}).T
for naam in voorspellingen:
    resultaten.loc[naam, "trainingstijd (s)"] = runtimes[f"training {naam}"]
    resultaten.loc[naam, "voorspeltijd (s)"] = runtimes[f"voorspellen {naam}"]
resultaten = resultaten.round(1)
resultaten.to_csv(RESULTS_DIR / "12_nhits_patchtst_huis_cons.csv")
resultaten

In [ ]:
plot_mae_per_stap(alle, echt_test)

In [ ]:
plot_voorspelling(voorspellingen, idx=0)
plot_voorspelling(voorspellingen, idx=len(test_origins) // 2)   # een ander moment

### Runtime overzicht
---

In [ ]:
runtimes["totaal notebook"] = time.perf_counter() - start_notebook
runtime_df = pd.DataFrame({"seconden": runtimes}).assign(minuten=lambda d: d["seconden"] / 60).round(2)
runtime_df